In [3]:
import torch
import torch.nn.functional as F

In [2]:
# classifier.py

# 이미지 4개, 각 이미지 길이 2 
X = torch.tensor([
    [2.0, 1.0],
    [1.0, 2.0],
    [-1.0, 1.0],
    [0.0, -1.0]
])

# 클래스 3개, 피쳐 2개
W = torch.tensor([
    [1.0, 0.0, -1.0],
    [0.0, 1.0, 1.0],
], requires_grad=True)

b = torch.zeros(3,requires_grad=True)

logits = X @ W + b
print(logits)

tensor([[ 2.,  1., -1.],
        [ 1.,  2.,  1.],
        [-1.,  1.,  2.],
        [ 0., -1., -1.]], grad_fn=<AddBackward0>)


In [ ]:
target = torch.tensor([0, 1, 2, 1])
loss = F.cross_entropy(logits, target) # 크로스 엔트로피가 평균을 낸 로스
loss_no_mean = F.cross_entropy(logits, target, reduction="none") # 일반적으로는 크로스엔트로피에서 평균을 낸다

print(loss)
print()
print(loss_no_mean)

tensor(0.7002, grad_fn=<NllLossBackward0>)

tensor([0.3490, 0.5514, 0.3490, 1.5514], grad_fn=<NllLossBackward0>)


In [ ]:
# backpropagation
loss = loss_no_mean.mean()

loss.backward()

print(f"W.grad = {0}", W.grad)
print(f"b.grad : {0}", b.grad)


W.grad = 0 tensor([[-0.1031, -0.0411,  0.1442],
        [-0.1029,  0.1148, -0.0119]])
b.grad : 0 tensor([ 0.1321, -0.1732,  0.0411])


In [7]:
# updating gradient 

lr = 0.1 

with torch.no_grad():
    W -= lr * W.grad
    b -= lr * b.grad

logits_after = X @ W + b
loss_after = F.cross_entropy(logits_after, target)

print(f"Before : {0}",loss.item())
print(f"After : {0}",loss_after.item())

Before : 0 0.7002284526824951
After : 0 0.6896925568580627


In [13]:
# loop with AutoGrad

# input image 4, length 2
X = torch.tensor([
    [2.0, 1.0],
    [1.0, 2.0],
    [-1.0, 1.0],
    [0.0, -1.0]
])

# class 3, feature 2
W = torch.tensor([
    [1.0, 0.0, -1.0],
    [0.0, 1.0, 1.0],
], requires_grad=True)

b = torch.zeros(3,requires_grad=True)

# target 4 
target = torch.tensor([0, 1, 2, 1])

lr = 0.1

# iteration 5
print("-"*5+"INIT"+"-"*5)
for step in range(500):
    # forward
    logits = X @ W + b 
    losses = F.cross_entropy(logits,target)

    losses.backward()

    with torch.no_grad(): # ?
        W -= lr * W.grad
        b -= lr * b.grad
    
    W.grad = None
    b.grad = None

    print(f"STEP : {step}, - LOSS : {losses.item()}")

print("-"*5+"TERMINATE"+"-"*5)


-----INIT-----
STEP : 0, - LOSS : 0.7002283930778503
STEP : 1, - LOSS : 0.6896925568580627
STEP : 2, - LOSS : 0.6794836521148682
STEP : 3, - LOSS : 0.6695886850357056
STEP : 4, - LOSS : 0.6599951982498169
STEP : 5, - LOSS : 0.6506918668746948
STEP : 6, - LOSS : 0.6416680216789246
STEP : 7, - LOSS : 0.6329137086868286
STEP : 8, - LOSS : 0.6244191527366638
STEP : 9, - LOSS : 0.6161752939224243
STEP : 10, - LOSS : 0.6081734299659729
STEP : 11, - LOSS : 0.600405216217041
STEP : 12, - LOSS : 0.5928627252578735
STEP : 13, - LOSS : 0.5855382084846497
STEP : 14, - LOSS : 0.5784240961074829
STEP : 15, - LOSS : 0.571513295173645
STEP : 16, - LOSS : 0.564798891544342
STEP : 17, - LOSS : 0.5582740306854248
STEP : 18, - LOSS : 0.5519322156906128
STEP : 19, - LOSS : 0.5457670092582703
STEP : 20, - LOSS : 0.5397723317146301
STEP : 21, - LOSS : 0.5339422225952148
STEP : 22, - LOSS : 0.5282707810401917
STEP : 23, - LOSS : 0.5227525234222412
STEP : 24, - LOSS : 0.5173819065093994
STEP : 25, - LOSS : 0.5

In [14]:
# checking performance 

with torch.no_grad():
    final_logits = X @ W + b
    prediction = final_logits.argmax(dim=1)


    print("final logits:\n", final_logits)
    print("prediction:", prediction)    
    print("target:    ", target)


final logits:
 tensor([[ 4.8242,  2.6527, -5.4769],
        [ 0.9933,  3.1598, -0.1531],
        [-5.5467,  2.4498,  5.0968],
        [-1.3417,  1.3679, -2.0262]])
prediction: tensor([0, 1, 2, 1])
target:     tensor([0, 1, 2, 1])
